# Lab 7 - エージェントを作り、つないで、試してから Hosted Agent にする

この Notebook では Microsoft Agent Framework のエージェントと workflow を**自分で組み立てます**。完成済みの `run_workflow()` を呼ぶだけではなく、誰が何を受け取り、どこへ渡すかをコード・図・途中回答で確認します。

**使用する kernel:** `Python (Foundry Hosted Agent)`。Lab 1 のセットアップと `az login` を完了してから、上から 1 cell ずつ進めてください。

| 順番 | 作るもの・確認するもの | Azure の呼び出し |
|---|---|---|
| 1–2 | 接続設定と、役割の違う 3 つのエージェント | モデル呼び出しなし |
| 3–4 | 順番に処理する workflow と、その実物から生成したグラフ | なし |
| 5–6 | 途中回答を見ながら実行し、依頼を変えてテスト | Foundry のモデルを利用 |
| 7 | fake client による、Azure を使わない構造テスト | なし |
| 8 | 同じ構成の Python source を Hosted Agent にデプロイする方法 | Terminal で明示的に実行 |

> **境界と料金:** Notebook 上でローカル実行する場合も、モデル推論は Foundry で行われ、入力と途中回答が送信されます。架空のデータだけを使ってください。標準の 1 依頼で 3 回、追加テスト 2 件でさらに 6 回のエージェント呼び出しがあり、モデル利用料金が発生します。予約・承認・Travel Ops API・Foundry IQ は呼びません。Hosted Agent / source remote build にはプレビューの制約と別途の実行料金があります。Run All ではデプロイしません。

## 1. Repository と Foundry project を読み込む

Lab 1 が生成した `.workshop/context.json` から project endpoint と model deployment 名を取得します。`.env` を手で編集する必要はありません。

この段階では設定を読むだけです。Hosted Agent の登録やモデルへの質問はまだ行いません。

In [ ]:
import json
import os
import sys
from pathlib import Path


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "hosted-agent" / "workflow.py").is_file():
            return candidate
    raise RuntimeError(
        "Repository root が見つかりません。Codespace でこの Notebook を開いてください。"
    )


REPO_ROOT = find_repo_root()
context_path = REPO_ROOT / ".workshop" / "context.json"
if not context_path.is_file():
    raise FileNotFoundError(
        "Lab 1 のセットアップを完了し、.workshop/context.json を作成してください。"
    )
context = json.loads(context_path.read_text(encoding="utf-8"))
outputs = context["terraform_outputs"]

os.environ["FOUNDRY_PROJECT_ENDPOINT"] = outputs["foundry_project_endpoint"]["value"]
os.environ["FOUNDRY_MODEL"] = outputs["primary_model_deployment_name"]["value"]

hosted_source = REPO_ROOT / "src" / "hosted-agent"
if str(hosted_source) not in sys.path:
    sys.path.insert(0, str(hosted_source))

print(f"Project: {outputs['foundry_project_name']['value']}")
print(f"Model deployment: {os.environ['FOUNDRY_MODEL']}")

## 2. Agent Framework でエージェントを作る

エージェントは、この例では **chat client + 名前 + instructions** です。同じモデルでも instructions を分けると担当する仕事が変わります。`as_agent()` は Python のオブジェクトを作る操作で、Foundry Portal に 3 つの agent を登録する操作ではありません。

| Agent | 受け取る情報 | 次へ渡す回答 |
|---|---|---|
| `policy_agent` | 元の出張依頼 | 不足情報・規程上の注意点 |
| `planner_agent` | 元の依頼 + policy の回答 | 食事・宿泊の概算と次のアクション |
| `reviewer_agent` | 元の依頼 + policy / planner の回答 | 矛盾を修正した最終回答 |

接続と架空の規程・instructions は `src/hosted-agent/workflow.py` から再利用します。まず、この演習で参照する規程を読んでください。過去の Lab で作った Toolbox や検索への接続は、この workflow にはありません。

In [ ]:
import workflow

chat_client = workflow.create_chat_client()
print(workflow.WORKSHOP_POLICY)

### 2-1. 規程を確認する `policy_agent`

必要情報が足りているか、規程で注意すべきことは何かを整理します。次の cell は agent を作り、その agent に実際に設定された instructions を表示します。**まだ推論は実行しません。**

In [ ]:
policy_agent = chat_client.as_agent(
    name="policy_agent",
    instructions=workflow.POLICY_AGENT_INSTRUCTIONS,
)
print(f"Created: {policy_agent.name}")
print(policy_agent.default_options["instructions"])

### 2-2. 出張案を作る `planner_agent`

規程確認を受けて食事・宿泊の概算を作ります。不足情報がある場合は推測して計算しません。前の回答を渡す仕組みは agent 自体ではなく、次の節で構築する workflow が担当します。

In [ ]:
planner_agent = chat_client.as_agent(
    name="planner_agent",
    instructions=workflow.PLANNER_AGENT_INSTRUCTIONS,
)
print(f"Created: {planner_agent.name}")
print(planner_agent.default_options["instructions"])

### 2-3. 回答を仕上げる `reviewer_agent`

前の 2 人の回答を見て、矛盾や架空の航空券価格を修正します。LLM に修正を依頼するだけでは正しさの保証にはならないため、後で実際の回答を観察・テストします。

In [ ]:
reviewer_agent = chat_client.as_agent(
    name="reviewer_agent",
    instructions=workflow.REVIEWER_AGENT_INSTRUCTIONS,
)
print(f"Created: {reviewer_agent.name}")
print(reviewer_agent.default_options["instructions"])

## 3. 作ったエージェントを workflow につなぐ

`SequentialBuilder(participants=[...])` に**実行する順番でオブジェクトを並べます**。元の依頼と、それまでの agent の回答が会話として次へ渡されます。並列実行や、モデルが次の担当者を選ぶ方式ではありません。

最後の `reviewer_agent` の回答が、そのまま workflow の最終回答です。simulation の注意書きも reviewer の instructions で指示しています。Python による自動補完は行わないため、後の cell で実際の回答に含まれるか確認します。

`output_from=[reviewer_agent]` で最終回答を返す担当を明示し、`intermediate_output_from="all_other"` で policy / planner の途中回答を観察できるようにします。前者は `output`、後者は `intermediate` イベントです。

組み立て部分を `build_travel_workflow()` として定義します。後のテストでは、この同じ定義から新しい workflow を作り、前の依頼の会話を持ち越さないようにします。ここでも推論はまだ実行しません。

In [ ]:
from agent_framework import Workflow
from agent_framework.orchestrations import SequentialBuilder

participants = [policy_agent, planner_agent, reviewer_agent]


def build_travel_workflow() -> Workflow:
    return SequentialBuilder(
        participants=participants,
        output_from=[reviewer_agent],
        intermediate_output_from="all_other",
    ).build()


travel_workflow = build_travel_workflow()
expected_order = [agent.name for agent in participants]
print(" -> ".join(expected_order))

## 4. 作った workflow をグラフで可視化する

`WorkflowViz(travel_workflow)` は、説明用の手書きの図ではなく**今作った workflow のノードと接続**から図を生成します。`policy_agent -> planner_agent -> reviewer_agent` の順につながっているか確認してください。表示される入力変換などの補助ノードは framework が追加するもので、LLM ではありません。

Codespace には Graphviz をセットアップ済みです。DOT をローカルで SVG に変換して Notebook 内に表示します。外部の描画サービスや CDN にグラフを送信しません。以前作った Codespace で `dot` がない場合は、Terminal で `sudo apt-get update && sudo apt-get install -y graphviz` を実行してこの cell を再実行してください。未導入の場合も Mermaid の定義を表示しますが、文字列だけではグラフ表示の完了ではありません。

In [ ]:
import shutil
import subprocess

from agent_framework import WorkflowViz
from IPython.display import SVG, Code, Markdown, display

viz = WorkflowViz(travel_workflow)
mermaid_graph = viz.to_mermaid()
dot_executable = shutil.which("dot")
if dot_executable is None:
    print("グラフ画像は未生成です。Graphviz を導入し、この cell を再実行してください。")
    display(Code(mermaid_graph, language="text"))
else:
    rendered = subprocess.run(
        [dot_executable, "-Tsvg"],
        input=viz.to_digraph(),
        capture_output=True,
        text=True,
        encoding="utf-8",
        check=False,
    )
    if rendered.returncode != 0:
        print(rendered.stderr)
        rendered.check_returncode()
    display(SVG(data=rendered.stdout))

## 5. Notebook 上で実行し、各 agent の回答を観察する

ここから Foundry のモデルを呼びます。最初は標準の依頼で実行してください。`user_request` を変えると、自分の架空の依頼でも試せます。

`travel_workflow.run(..., stream=True)` のイベントを読み、開始した担当者、policy / planner の途中回答、reviewer の最終回答を分けて表示します。どの agent の回答も `AgentResponseUpdate` という断片で届くので連結します。最終回答は reviewer の断片をすべて受け取ってから表示します。Notebook では `asyncio.run()` ではなくトップレベルの `await` / `async for` を使います。

In [ ]:
user_request = workflow.SAMPLE_REQUEST
print(user_request)

In [ ]:
from agent_framework import AgentResponse, AgentResponseUpdate

execution_order = []
intermediate_answers = {}
final_chunks = []
answer = None
travel_workflow = build_travel_workflow()

async for event in travel_workflow.run(user_request, stream=True):
    if event.type in {"failed", "executor_failed", "error"}:
        raise RuntimeError(f"Workflow failed: {event.details or event.data}")
    if event.type == "executor_invoked" and event.executor_id in expected_order:
        execution_order.append(event.executor_id)
        print(f"\n開始: {event.executor_id}", flush=True)
    elif event.type == "intermediate" and isinstance(event.data, AgentResponseUpdate):
        if event.executor_id not in intermediate_answers:
            intermediate_answers[event.executor_id] = ""
            print(f"\n--- {event.executor_id} の途中回答 ---", flush=True)
        intermediate_answers[event.executor_id] += event.data.text
        print(event.data.text, end="", flush=True)
    elif event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        final_chunks.append(event.data.text)

if not "".join(final_chunks).strip():
    raise RuntimeError("reviewer の最終回答が空です。実行ログを確認してください。")
answer = "".join(final_chunks)
display(Markdown("### reviewer_agent の最終回答"))
print(answer)

### 5-1. 実行順序と最終出力を確認する

次の assert は順序・途中回答の存在・出力形式・必須注意書きを確認します。**内容の正しさまで自動で判定するテストではありません。** 標準の依頼では、表示された途中回答と最終回答を次の観点で読み比べてください。

| 観点 | 標準の依頼で期待する内容 |
|---|---|
| policy | 必要情報が揃っており、国内 economy の規程を確認している |
| planner | 2 日分の食事 6,000 円、1 泊の宿泊上限 15,000 円、航空券は要見積もり |
| reviewer | 宿泊上限を使った、航空券を含まない小計は 21,000 円。予約・承認済みと表現していない |
| 注意書き | reviewer が instructions に従い、simulation の注意書きを含めている |

違いがあれば、どの agent の回答からずれたかを追ってください。注意書きは自動補完しないので、省略された場合は assert が失敗します。入力を変更した場合、金額などの期待値も変わります。

In [ ]:
assert execution_order == expected_order, execution_order
assert set(intermediate_answers) == set(expected_order[:-1]), intermediate_answers.keys()
assert all(intermediate_answers.values()), "空の途中回答があります。"
assert all(heading in answer for heading in ["規程確認", "概算", "次のアクション"])
assert workflow.SIMULATION_NOTICE in answer
print("OK: 順序・途中回答・最終回答の形式・simulation 注意書きを確認しました。")

## 6. 入力を変えて Notebook 上でテストする

正常系の 1 件だけでは不十分です。情報が足りない依頼と、海外 business の依頼を、**この Notebook で定義した workflow** に渡します。`build_travel_workflow()` でケースごとに新しく組み立て、前のテストの会話を持ち越さないようにします。同じ workflow インスタンスを使い回すと会話が残るため、入力不足のテストで前の依頼の情報が補われてしまいます。

この cell は追加で 2 件、合計 6 回のエージェント呼び出しを行います。期待する振る舞いと実際の回答を読み比べてください。ここでの assert は出力と注意書きだけの確認であり、期待する判断ができたかは目視します。回答が期待と違えば、`user_request` に同じ依頼を入れ、第 5 節を再実行すると途中回答を調べられます。

In [ ]:
test_cases = [
    {
        "name": "入力不足",
        "request": "大阪へ出張したいです。概算を作ってください。",
        "expected": "出発地・日程・座席クラス・目的などを確認し、不足情報を推測して計算しない。",
    },
    {
        "name": "海外・business",
        "request": (
            "2026年9月10日から12日まで、東京からシアトルへ1名で社内会議に行きます。"
            "座席クラスは business です。規程確認と概算を作ってください。"
        ),
        "expected": "マネージャーと部門 VP の事前確認を案内し、国内の日当・宿泊上限を流用しない。",
    },
]

case_results = {}
for case in test_cases:
    display(Markdown(f"### {case['name']}\n\n期待する振る舞い: {case['expected']}"))
    print(f"入力: {case['request']}", flush=True)
    case_workflow = build_travel_workflow()
    result = await case_workflow.run(case["request"])
    case_outputs = result.get_outputs()
    assert len(case_outputs) == 1 and isinstance(case_outputs[0], AgentResponse)
    case_answer = case_outputs[0].text
    assert case_answer and workflow.SIMULATION_NOTICE in case_answer
    case_results[case["name"]] = case_answer
    print(case_answer)

## 7. Azure を使わない contract test を実行する

今までの実モデルによるテストと、次の構造テストは目的が違います。fake chat client は固定の回答を返すため、**モデルの判断品質は評価できません**。代わりに、実際の Agent Framework を使って、順序、元の依頼と回答の引き継ぎ、reviewer の回答をそのまま返すこと、Notebook とデプロイ用 source の構成の一致を再現可能に確認します。

このテストはディスク上のファイルを読みます。Notebook を編集した場合は先に保存してください。失敗したら deploy に進まず、表示されたテスト名とエラーを確認します。

In [ ]:
completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        str(REPO_ROOT / "tests" / "contract" / "hosted_agent"),
        "-q",
    ],
    cwd=REPO_ROOT,
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    env={**os.environ, "PYTHONUTF8": "1"},
)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    completed.check_returncode()

## 8. ローカルで理解した workflow を Hosted Agent にする

ここまでは Notebook の Python プロセスが orchestration を実行していました。Hosted Agent にすると、この Python の処理を Foundry の実行基盤がホストします。3 つの agent をそれぞれ deploy するのではなく、workflow 全体を 1 つの Hosted Agent として公開します。

### 8-1. デプロイされるコードとの対応を確認する

| Notebook で行ったこと | デプロイ用 source |
|---|---|
| `chat_client.as_agent()` を 3 回呼ぶ | `workflow.py` の `build_workflow()` |
| `SequentialBuilder` で順番に接続する | 同じ `build_workflow()` 内の participants |
| 規程・instructions (注意書きの指示を含む) | Notebook と共有している `workflow.py` の定数 |
| `travel_workflow.run()` で直接呼ぶ | `main.py` が `build_workflow().as_agent(...)` を `ResponsesHostServer` で公開 |
| 途中回答を表示する | Notebook だけの観察設定。ホストは最終回答だけを返す |

**Notebook 自体やメモリ上の変更はデプロイされません。** 学習用に agent 作成と接続を cell に展開しています。第 2–3 節で名前・instructions・順序を実験的に変え、それを deploy したい場合は `src/hosted-agent/workflow.py` にも反映し、Notebook のカーネルを再起動して上から再実行・テストしてください。通常の手順では source を変更する必要はありません。

次に表示するのは、説明用の抜粋ではなく、現在のデプロイ用 `build_workflow()` の実際の定義です。学習用の出力設定 (`output_from` / `intermediate_output_from`) を除いて、同じ組み立て方になっていることを確認します。

In [ ]:
import inspect

display(Code(inspect.getsource(workflow.build_workflow), language="python"))

### 8-2. Terminal から明示的に deploy する

第 5–7 節で期待する回答とテスト結果を確認してから、**repository root の Terminal** で次を実行します。Notebook の kernel とは異なる **root の `.venv`** を使います。Hosted runtime と deploy SDK は `azure-ai-projects` の互換バージョンが異なるため、環境を混ぜません。

```bash
.venv/bin/python scripts/deploy_hosted_agent.py --output json
```

この script は `src/hosted-agent/` の source を zip 化し、Python 3.13 の remote build を開始し、version が `active` または `failed` になるまで有限時間で待ちます。Docker、ACR、追加の sign-in は不要です。実行ごとに不変の agent version を作成します。

`status: "active"` を確認したら、[Lab 7 の Portal で実行する手順](../labs/07-hosted-multi-agent.md#3-portal-で実行する)に戻り、`contoso-travel-hosted-planner` の Playground に標準の依頼を送ってください。Notebook と同じ役割・順序で動きますが、モデルの回答文は毎回一致するとは限りません。終了後は [Lab 8](../labs/08-observability-cleanup.md) で trace と cleanup を確認します。

### 参考資料

- [Microsoft Learn: Workflow visualization](https://learn.microsoft.com/en-us/agent-framework/workflows/visualization)
- [公式 Agent Framework orchestrations](https://github.com/microsoft/agent-framework/tree/main/python/packages/orchestrations)

この Notebook の API は `requirements.txt` の `agent-framework-core==1.15.0` / `agent-framework-orchestrations==1.1.1` を対象にしています。公式資料の参照日: 2026-09-06。